# Task 5: O*NET 30.3 file preparation

This notebook documents and runs the preparation of O*NET data for the project's master occupation table. The outputs use six-digit 2018 SOC codes so they can later be joined to the three AI-risk indices.

The reusable processing logic lives in `task5_onet_prep.py`. Keeping it in a script means the same pipeline can be rerun outside Jupyter and avoids maintaining two conflicting copies of the transformation code.

## 1. Setup and source inventory

The raw workbooks are left unchanged in `data/datasets/`. This notebook only reads them and writes new files to `data/datasets/processed/`.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/datasets")
PROCESSED_DIR = DATA_DIR / "processed"

onet_files = sorted(DATA_DIR.glob("onet_*.xlsx"))
source_inventory = []

for file in onet_files:
    excel = pd.ExcelFile(file)
    preview = pd.read_excel(file, sheet_name=excel.sheet_names[0], nrows=2)
    source_inventory.append({
        "file": file.name,
        "sheet": excel.sheet_names[0],
        "columns": len(preview.columns),
    })

pd.DataFrame(source_inventory)

,file,sheet,columns
0,onet_abilities.xlsx,Abilities,15
1,onet_essential_skills.xlsx,Essential Skills,15
2,onet_job_zones.xlsx,Job Zones,5
3,onet_software_skills.xlsx,Software Skills,7
4,onet_transferable_skills.xlsx,Transferable Skills,15
5,onet_work_activities.xlsx,Work Activities,15
6,onet_work_context.xlsx,Work Context,16


## 2. Check the Job Zones rollup

O*NET Job Zones are already numeric categories from 1 to 5, but raw records are at the detailed eight-digit O*NET-SOC level. Before collapsing codes such as `11-1011.00` and `11-1011.03` to `11-1011`, we check whether detailed occupations disagree on their Job Zone.

In [2]:
job_zones_raw = pd.read_excel(DATA_DIR / "onet_job_zones.xlsx", sheet_name="Job Zones")
job_zones_raw["soc_6digit"] = job_zones_raw["O*NET-SOC Code"].str.split(".").str[0]

job_zone_variation = job_zones_raw.groupby("soc_6digit")["Job Zone"].nunique()

print("Raw Job Zone rows:", len(job_zones_raw))
print("Unique six-digit SOC codes:", job_zones_raw["soc_6digit"].nunique())
print("SOC families with conflicting Job Zones:", int((job_zone_variation > 1).sum()))

job_zone_variation.value_counts().sort_index()

Raw Job Zone rows: 923
Unique six-digit SOC codes: 798
SOC families with conflicting Job Zones: 27


Job Zone
1    771
2     26
3      1
Name: count, dtype: int64

## 3. Preparation decisions

- Use the `IM` (Importance) scale for Abilities, Skills, and Work Activities.
- Combine Essential Skills (`2.A` O*NET elements) and Transferable Skills (`2.B` elements) into the project's one Skills table.
- Use `CX` continuous ratings for Work Context; exclude category-distribution scales.
- Average detailed O*NET occupations within each six-digit SOC and feature, because no employment weights are supplied.
- Use a `.00` base Job Zone when one exists. For SOC families without a base row, use the modal detailed Job Zone; ties use the lower tied category and are explicitly flagged.

The full rationale and naming convention are in `../data/datasets/processed/onet_prep_methods.md`.

## 4. Run the reproducible preparation pipeline

Running the next cell rebuilds the five clean CSVs and the feature dictionary. It does not modify the raw Excel workbooks.

In [3]:
from runpy import run_path

run_path("task5_onet_prep.py", run_name="__main__")

Wrote data/datasets/processed/onet_job_zones_6digit.csv: 798 rows, 7 columns
  Job Zone conflicts retained in audit flags: 27


Wrote data/datasets/processed/onet_abilities_6digit.csv: 774 rows, 55 columns


Wrote data/datasets/processed/onet_skills_6digit.csv: 774 rows, 38 columns


Wrote data/datasets/processed/onet_work_activities_6digit.csv: 774 rows, 44 columns


Wrote data/datasets/processed/onet_work_context_6digit.csv: 774 rows, 58 columns
Wrote data/datasets/processed/onet_feature_dictionary.csv: 183 rows, 7 columns


{'__name__': '__main__',
 '__doc__': 'Prepare O*NET 30.3 occupation features for the Task 5 master join.\n\nRun from the repository root with:\n    .venv-1/bin/python notebooks/task5_onet_prep.py\n\nThe script preserves the raw files, writes five six-digit-SOC CSVs to\ndata/datasets/processed/, and records the feature-to-source mapping.\n',
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': 'task5_onet_prep.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_im

## 5. Verify the five clean output tables

Each output must have one unique row per six-digit SOC. Numeric feature tables are allowed to have fewer SOCs than Job Zones because source coverage differs; those gaps are documented for the later coverage report.

In [4]:
output_files = [
    "onet_job_zones_6digit.csv",
    "onet_abilities_6digit.csv",
    "onet_skills_6digit.csv",
    "onet_work_activities_6digit.csv",
    "onet_work_context_6digit.csv",
]

quality_summary = []
for filename in output_files:
    table = pd.read_csv(PROCESSED_DIR / filename, dtype={"soc_6digit": "string"})
    metadata_columns = {
        "soc_6digit", "title", "title_rollup_method", "job_zone",
        "job_zone_rollup_method", "job_zone_had_conflict",
        "job_zone_source_value_count",
    }
    feature_columns = [column for column in table.columns if column not in metadata_columns]
    quality_summary.append({
        "file": filename,
        "rows": len(table),
        "unique_soc_keys": table["soc_6digit"].nunique(),
        "feature_columns": len(feature_columns),
        "missing_feature_values": int(table[feature_columns].isna().sum().sum()) if feature_columns else 0,
    })

pd.DataFrame(quality_summary)

,file,rows,unique_soc_keys,feature_columns,missing_feature_values
0,onet_job_zones_6digit.csv,798,798,0,0
1,onet_abilities_6digit.csv,774,774,52,0
2,onet_skills_6digit.csv,774,774,35,0
3,onet_work_activities_6digit.csv,774,774,41,0
4,onet_work_context_6digit.csv,774,774,55,0


## Result

Task 5 produces five join-ready O*NET tables and a feature dictionary. The next team step is to join these tables to the AI-risk indices in Task 6, retaining source-specific coverage counts in the project coverage report.